# CALM — Adaptive Multimodal Wildfire Monitoring

Demo hệ thống AI giám sát cháy rừng: Planning → Routing → Execution → Evaluation.

- **Plan-driven:** PlanningAgent → RouterAgent → ExecutionAgent
- **Agents:** DataKnowledge, Prediction (SeasFire/heuristic), QA, RSEN, Evaluator

## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path("/home/hungvu/code/khanh/v2").resolve()
CALM_DIR = ROOT / "calm-"
SEASFIRE_DIR = ROOT / "seasfire-ml"
DATACUBE_DIR = ROOT / "seasfire-datacube"
DATA_DIR = ROOT / "data"

# Nếu file .env nằm trong calm-
ENV_PATH = CALM_DIR / ".env"

def run(cmd: str, allow_fail: bool = False):
    print(f"\n>>> {cmd}")
    rc = os.system(cmd)
    exit_code = rc >> 8
    if exit_code != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {exit_code}: {cmd}")
    return exit_code

print("ROOT =", ROOT)
print("CALM_DIR =", CALM_DIR)
print("ENV_PATH =", ENV_PATH)
print("Python =", sys.version)

# kiểm tra repo CALM
if not (CALM_DIR / "pyproject.toml").exists():
    raise FileNotFoundError(f"Không thấy pyproject.toml trong {CALM_DIR}")

DATA_DIR.mkdir(parents=True, exist_ok=True)

# clone seasfire-ml nếu chưa có
if not SEASFIRE_DIR.exists():
    run(f'git clone https://github.com/SeasFire/seasfire-ml.git "{SEASFIRE_DIR}"')
else:
    print("seasfire-ml đã có, bỏ qua clone")

# clone seasfire-datacube nếu chưa có
if not DATACUBE_DIR.exists():
    run(f'git clone https://github.com/SeasFire/seasfire-datacube.git "{DATACUBE_DIR}"')
else:
    print("seasfire-datacube đã có, bỏ qua clone")

# cài CALM bằng pip của đúng kernel hiện tại
run(f'cd "{CALM_DIR}" && "{sys.executable}" -m pip install -r requirements.txt')
run(f'cd "{CALM_DIR}" && "{sys.executable}" -m pip install -e .')

# add src vào sys.path trước khi import calm
CALM_SRC = CALM_DIR / "src"
if str(CALM_SRC) not in sys.path:
    sys.path.insert(0, str(CALM_SRC))

# load .env
if ENV_PATH.exists():
    try:
        # ưu tiên loader của project CALM
        from calm.utils.env_loader import load_env
        load_env(ENV_PATH)
        print(f"Đã load .env bằng calm.utils.env_loader: {ENV_PATH}")
    except Exception as e:
        print(f"load_env của CALM lỗi ({e}), thử fallback python-dotenv...")
        try:
            from dotenv import load_dotenv
        except ImportError:
            run(f'"{sys.executable}" -m pip install python-dotenv')
            from dotenv import load_dotenv
        load_dotenv(ENV_PATH, override=True)
        print(f"Đã load .env bằng python-dotenv: {ENV_PATH}")
else:
    print(f"Không thấy file .env tại {ENV_PATH}")

# cài seasfire-ml chỉ khi Python <= 3.10
py_major, py_minor = sys.version_info[:2]
if (SEASFIRE_DIR / "requirements.txt").exists():
    if (py_major, py_minor) <= (3, 10):
        run(f'cd "{SEASFIRE_DIR}" && "{sys.executable}" -m pip install -r requirements.txt')
    else:
        print(
            f"Bỏ qua cài requirements của seasfire-ml vì Python {py_major}.{py_minor}; "
            "repo này pin torch cũ."
        )
else:
    print("Không thấy requirements.txt trong seasfire-ml, bỏ qua")

# set mặc định nếu .env chưa có
os.environ.setdefault("SEASFIRE_ML_PATH", str(SEASFIRE_DIR))
os.environ.setdefault("SEASFIRE_DATASET_PATH", str(DATA_DIR))

# in ra các biến quan trọng đã load
keys_to_show = [
    "OPENROUTER_API_KEY",
    "OPENAI_API_KEY",
    "ENABLE_GEE",
    "ENABLE_COPERNICUS",
    "ENABLE_WEB_SEARCH",
    "ENABLE_ARXIV",
    "GEE_PROJECT_ID",
    "COPERNICUS_API_KEY",
    "COPERNICUS_URL",
    "MAX_NEWS_RESULTS",
    "MAX_ARXIV_PAPERS",
    "DEDUP_CHECK",
    "SEASFIRE_ML_PATH",
    "SEASFIRE_DATASET_PATH",
]

print("\n===== ENV CHECK =====")
for k in keys_to_show:
    v = os.environ.get(k)
    if v is None:
        print(f"{k} = <missing>")
    elif "KEY" in k or "TOKEN" in k or "SECRET" in k:
        masked = v[:6] + "..." + v[-4:] if len(v) > 12 else "***"
        print(f"{k} = {masked}")
    else:
        print(f"{k} = {v}")

print("\nSetup OK")

ROOT = /home/hungvu/code/khanh/v2
CALM_DIR = /home/hungvu/code/khanh/v2/calm-
ENV_PATH = /home/hungvu/code/khanh/v2/calm-/.env
Python = 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
seasfire-ml đã có, bỏ qua clone
seasfire-datacube đã có, bỏ qua clone

>>> cd "/home/hungvu/code/khanh/v2/calm-" && "/home/hungvu/code/khanh/.venv/bin/python" -m pip install -r requirements.txt



>>> cd "/home/hungvu/code/khanh/v2/calm-" && "/home/hungvu/code/khanh/.venv/bin/python" -m pip install -e .
Obtaining file:///home/hungvu/code/khanh/v2/calm-
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for calm (pyproject.toml): started
  Building editable for calm (pyproject.toml): finished with status 'done'
  Created wheel for calm: filename=calm-0.1.0-py3-none-any.whl size=1491 sha256=ddbd7bed10682898e45abcfbe4a5d

In [4]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    max_completion_tokens=10000,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
)

## 2. Planning Agent

URSA 3-node: Generator → Reflector → Formalizer. Output: plan JSON với step_id, action, agent, prompt.

In [5]:
from calm.agents.planning_agent import PlanningAgent

planner = PlanningAgent(llm=llm, config={}, n_max=3, f_max=3)
query = "Wildfire risk assessment for Amazon region next 7 days"
result = planner.invoke(query)

plan = result.get("final_output") or []
print("Approved:", result.get("approved"))
print("Steps:", len(plan))
for i, s in enumerate(plan, 1):
    print(f"  {i}. {s.get('step_id')} | {s.get('action')} | {s.get('agent')}")
    if s.get("prompt"):
        print(f"     prompt: {s['prompt'][:80]}...")

Approved: True
Steps: 6
  1. step-1 | web_search | qa
     prompt: Collect weather forecast, temperature, humidity, rainfall data, and assess curre...
  2. step-2 | retrieve_knowledge | data_knowledge
     prompt: Analyze historical wildfire data for trends in the Amazon region over the past y...
  3. step-3 | real_time_monitoring | data_knowledge
     prompt: Integrate real-time wildfire incident data and risk factors for the Amazon regio...
  4. step-4 | risk_assessment | prediction
     prompt: Evaluate wildfire risk for the Amazon region using the compiled data....
  5. step-5 | stakeholder_engagement | execution
     prompt: Communicate the findings of the wildfire risk assessment to local authorities, c...
  6. step-6 | data_security | execution
     prompt: Ensure that all data accessing during the assessment complies with security prot...


## 3. Router Agent

Xác định task_type (qa | prediction | hybrid) từ plan + query.

In [6]:
from calm.agents.router_agent import RouterAgent

router = RouterAgent(llm=llm, config={})
routing = router.route(query, plan)
print("Task type:", routing.task_type)
print("Confidence:", routing.confidence)
print("Required artifacts:", routing.required_artifacts)

Task type: prediction
Confidence: 0.9
Required artifacts: ['prediction', 'met_data', 'spatial_data']


## 4. DataKnowledgeAgent

Collect (GEE, CDS, Web, ArXiv) → Extract → Retrieve (ChromaDB). GeocodingTool chuyển địa chỉ văn bản → lat, lon.

In [7]:
from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.memory_agent import MemoryAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool
from calm.tools.geocoding import GeocodingTool
from calm.tools.earth_engine import EarthEngineTool
from calm.tools.copernicus import CopernicusTool
from calm.tools.arxiv_tool import ArXivTool

safety = SafetyChecker(llm=llm)
tools = {
    "earth_engine": EarthEngineTool(safety_checker=safety, config={"gee_project": os.environ.get("GEE_PROJECT_ID")}),
    "copernicus": CopernicusTool(safety_checker=safety, config={}),
    "web_search": WebSearchTool(safety_checker=safety, config={"max_news_results": 5}),
    "arxiv": ArXivTool(safety_checker=safety, config={"max_arxiv_papers": 3}),
    "geocoding": GeocodingTool(safety_checker=safety, config={}),
}

memory = MemoryAgent(long_term_store=ChromaMemoryStore(collection_name="calm_demo", persist_directory=".chroma", k=3))
data_agent = DataKnowledgeAgent(llm=llm, tools=tools, memory_store=memory, config={"dedup_check": False})

params = {"location": "California, USA", "time_range": {"start": "2024-01-01", "end": "2024-12-31"}}
ret = data_agent.retrieve("wildfire causes in California 2024", parameters=params)
print("Retrieved:", len(ret.get("retrieved_data", [])), "sources")
print("Dedup hit:", ret.get("dedup", False))

GEE collection failed: [UNSAFE] Safety check blocked: GEE fetch_satellite_stats location={'lat': 36.7014631, 'lon': -118.755997} time_range={'start': '2024-01-01', 'end': '2024-12-31'} product=LANDSAT/LC08/C02/T1_L2
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12328.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieved: 14 sources
Dedup hit: False


## 5. RSEN — Validation

Weather Analyst + Geo Analyst (song song) → Ops Coordinator → Plausible/Implausible.

In [8]:
from calm.agents.rsen_module import RSENModule

rsen = RSENModule(llm=llm, memory_store=memory, k=3)
val = rsen.validate(
    prediction={"risk_level": "High", "confidence": 0.8},
    met_data={"temperature": 35.0, "humidity": 0.2},
    spatial_data={"fuel_type": "Shrubland", "slope": 25},
)
print("Decision:", val.get("validation_decision"))
print("Rationale:", (val.get("final_rationale") or "")[:200]+"...")

Decision: Plausible
Rationale: 1. The initial prediction of high risk is supported by the Weather Analyst's findings: high temperatures and low humidity create dry conditions that enhance fire risk.
2. The Weather Report indicates ...


## 6. Wildfire QA Agent

Evidence Evaluator → retrieve → trả lời với citations.

In [10]:
from calm.agents.qa_agent import WildfireQAAgent
from calm.tools.web_search import WebSearchTool
from calm.tools.safety_check import SafetyChecker
web_tool = globals().get("web_tool") or globals().get("web")
if web_tool is None:
    web_tool = (globals().get("tools") or {}).get("web_search")
if web_tool is None:
    safety = globals().get("safety") or SafetyChecker(llm=llm)
    web_tool = WebSearchTool(safety_checker=safety, config={"max_news_results": 5})


In [11]:
from calm.agents.qa_agent import WildfireQAAgent

qa = WildfireQAAgent(llm=llm, data_agent=data_agent, web_search_tool=web_tool, memory_store=memory, config={"evidence_threshold": 0.65})
qa_result = qa.invoke("What caused the 2023 Canadian wildfires?")
out = qa_result.get("final_output") or {}
print("Answer:", (out.get("answer", ""))[:300])
print("Citations:", out.get("citations", [])[:2])
print("Confidence:", out.get("confidence"))

Answer: The 2023 Canadian wildfires were caused by a combination of factors including climate change, human activities, and natural conditions. Specifically, significant contributors include:

1. **Climate Change**: The increasing temperatures and prolonged dry spells associated with climate change have cre
Citations: ['https://en.wikipedia.org/wiki/2023_Canadian_wildfires', 'https://thenarwhal.ca/wildfire-canada-explainer/']
Confidence: 0.95


## 7. Full Pipeline — CALMOrchestrator

Một lệnh `run(query)` → Planning → Routing → Execution. Tự route QA hoặc Prediction.

In [12]:
from calm.orchestrator import CALMOrchestrator
from calm.memory.chroma_store import ChromaMemoryStore

memory_store = ChromaMemoryStore(collection_name="calm_orch", persist_directory=".chroma")
# from_llm sẽ tự gắn đủ tool mặc định: GEE, Copernicus, Web, ArXiv, Geocoding
orch = CALMOrchestrator.from_llm(llm=llm, memory_store=memory_store, tools={}, config={"planner_n_max": 2})

# Mặc định chạy QA để demo ổn định trên mọi máy
result = orch.run("What causes wildfires in the Amazon rainforest?")
print("Task type:", result.get("task_type"))
if result.get("task_type") == "qa":
    print("Answer:", (result.get("answer", "") or "")[:250])
else:
    print("Risk:", result.get("risk_level"), "|", result.get("decision"), "|", (result.get("rationale", "") or "")[:150])



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8968.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Task type: qa
Answer: The documented causes of wildfires in the Amazon rainforest can be classified into two main categories: anthropogenic (human-induced) factors and natural factors. The primary anthropogenic causes include land clearing for agriculture, particularly th


In [13]:
pred = orch.run("Predict wildfire risk California next 7 days")
print(pred.get("risk_level"), pred.get("decision"), (pred.get("rationale", "") or "")[:150])
if result.get("error"):
    print("Error:", str(result["error"])[:150])

Unknown Unknown 


## 8. Evaluator — LLM-as-a-Judge

Chấm 5 tiêu chí (0–100): Data Accuracy, Explainability, Jargon Avoidance, Redundancy Avoidance, Citation Quality. Passing: average >= 75.

In [14]:
from calm.agents.evaluator_agent import EvaluatorAgent

evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})
eval_query = "What causes wildfires in the Amazon?"
orch_result = orch.run(eval_query)  
eval_result = evaluator.evaluate(response=orch_result, query=eval_query)

print("Average score:", eval_result.get("average_score"))
print("Passed:", eval_result.get("passed"))
print("Scores:", eval_result.get("scores", {}))

Average score: 53.0
Passed: False
Scores: {'data_accuracy': 40, 'explainability': 65, 'jargon_avoidance': 70, 'redundancy_avoidance': 50, 'citation_quality': 60}
